# 03 — GRPO loop against the live OpenEnv Space

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SnehShah/house-md-env/blob/main/notebooks/03_grpo.ipynb)
[![Adapter](https://huggingface.co/datasets/huggingface/badges/resolve/main/open-in-hf-models-sm.svg)](https://huggingface.co/SnehShah/house-md-grpo-optimized-gemma3-4b-v3)
[![W&B](https://img.shields.io/badge/W%26B-house--md-yellow)](https://wandb.ai/sneh2909-christ-university/house-md?nw=nwusersneh2909)

**Goal**: take the SFT checkpoint and teach it to *actually diagnose* via Group-Relative Policy Optimisation. SFT taught the model the action JSON shape; GRPO teaches it which action *sequences* are worth taking by scoring 4-rollout groups against the live OpenEnv environment.

**Hardware**: this mini-loop is sized for a Colab **T4** — 30 steps × group_size 4 finishes in ~25 min. The production curves at <https://wandb.ai/sneh2909-christ-university/house-md?nw=nwusersneh2909> ran 100 steps × group_size 8 on an A100.

**The full implementation is at [`scripts/train_grpo_optimized.py`](https://github.com/SnehShah/house-md-env/blob/main/scripts/train_grpo_optimized.py)** — this notebook is a *teaching mirror* of the same algorithm:

1. Sample one patient (Level-1 curriculum).
2. Roll out the policy `K` times against the **live `SnehShah/house-md-env` Space**.
3. Score each rollout with `compute_all` (the same five-rubric reward we ship in production).
4. Compute group-relative advantages: $A_i = (r_i - \bar r) / (\sigma_r + 10^{-8})$.
5. Take a **policy-gradient update** weighted by those advantages.
6. Push the resulting LoRA adapter to your HF account.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q huggingface_hub wandb requests pandas matplotlib
!pip install -q "openenv-core>=0.2.2"
!pip install -q "git+https://github.com/SnehShah/house-md-env.git"

## 2. Auth + config

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"]      = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY") or ""
except Exception:
    pass

HF_USERNAME   = os.environ.get("HF_USERNAME", "SnehShah")
HF_TOKEN      = os.environ.get("HF_TOKEN", "")
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    wandb.init(project="house-md", name="grpo-colab-mini", config={"steps": 30, "group_size": 4})

BASE_MODEL    = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit"
SFT_ADAPTER   = os.environ.get("SFT_ADAPTER", "SnehShah/house-md-sft-gemma3-4b")
OUTPUT_HUB_ID = f"{HF_USERNAME}/house-md-grpo-gemma3-4b-mini"
ENV_URL       = "https://snehshah-house-md-env.hf.space"
TOTAL_STEPS   = 30
GROUP_SIZE    = 4
MAX_TURNS     = 12
TEMPERATURE   = 0.9
LR            = 1e-5
MAX_NEW_TOK   = 150
LEVEL_1       = ["ectopic_pregnancy", "stemi", "appendicitis", "migraine", "viral_uri"]

## 3. Load Gemma 3 4B + SFT adapter (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN or None,
)

from peft import PeftModel
try:
    model = PeftModel.from_pretrained(model, SFT_ADAPTER, is_trainable=True, token=HF_TOKEN or None)
    print(f"✓ loaded SFT adapter: {SFT_ADAPTER}")
except Exception as e:
    print(f"⚠  could not load SFT adapter ({e}); falling back to fresh LoRA")
    model = FastLanguageModel.get_peft_model(
        model, r=16, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        lora_alpha=16, lora_dropout=0.05, bias="none",
        use_gradient_checkpointing="unsloth", random_state=42,
    )
model.train()
optim = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
print("✓ ready for GRPO")

## 4. Rollout against the live Space

Each rollout opens a session on the Space, plays up to `MAX_TURNS` actions, and returns:
* the full chat history with token-level log-probs for every action (needed for the gradient)
* the final reward dict and total

We stash log-probs as we sample so we never need a second forward pass.

In [ ]:
import json, random, requests
import torch.nn.functional as F

API = ENV_URL + "/api"

def _try_parse(raw):
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except Exception:
        return {"type": "DIAGNOSE", "argument": "viral_uri", "rationale": "parse-fallback"}

def rollout_once(disease, seed):
    s = requests.post(f"{API}/episodes", json={"disease": disease, "seed": seed}).json()
    sid = s["session_id"]
    chosen_token_logp = []
    
    for turn in range(MAX_TURNS):
        ep = requests.get(f"{API}/episodes/{sid}").json()
        if ep["observation"]["done"]:
            break
        prompt = ep["observation"]["prompt"]
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            return_tensors="pt", add_generation_prompt=True,
        ).to(model.device)
        out = model.generate(
            inputs, max_new_tokens=MAX_NEW_TOK, do_sample=True, temperature=TEMPERATURE,
            pad_token_id=tokenizer.eos_token_id, return_dict_in_generate=True, output_scores=True,
        )
        seq = out.sequences[0]
        gen_ids = seq[inputs.shape[-1]:]
        logps = []
        for t, score in enumerate(out.scores):
            tok = gen_ids[t]
            if tok == tokenizer.eos_token_id:
                break
            logps.append(F.log_softmax(score[0], dim=-1)[tok])
        chosen_token_logp.append(torch.stack(logps).sum() if logps else torch.tensor(0.0, device=model.device))
        raw = tokenizer.decode(gen_ids, skip_special_tokens=True)
        action = _try_parse(raw)
        try:
            requests.post(f"{API}/episodes/{sid}/actions", json={"action": action}, timeout=10)
        except Exception:
            break
    
    rewards = requests.get(f"{API}/episodes/{sid}/rewards").json()
    requests.delete(f"{API}/episodes/{sid}")
    total = float(rewards.get("total", 0.0))
    return chosen_token_logp, total, rewards

## 5. The GRPO update

Standard group-relative loss, no clipping (see `train_grpo_optimized.py` for the PPO-style version):

$$ \mathcal{L} = -\frac{1}{K}\sum_{i=1}^{K} A_i \sum_{t} \log \pi_\theta(a_t^{(i)}|s_t^{(i)}) $$

with $A_i = (r_i - \bar r) / (\sigma_r + 10^{-8})$. Rollouts above the group mean get pushed up, those below get pushed down.

In [ ]:
import statistics
from pathlib import Path
import csv

log_path = Path("reward_log.csv")
if log_path.exists():
    log_path.unlink()
with log_path.open("w", newline="") as fh:
    csv.writer(fh).writerow(["step", "disease", "mean_r", "std_r", "max_r", "loss"])

history = []
for step in range(1, TOTAL_STEPS + 1):
    disease = random.choice(LEVEL_1)
    seed_base = random.randint(0, 10_000)
    
    logps_per_rollout = []
    rewards_per_rollout = []
    for k in range(GROUP_SIZE):
        lp, r, _ = rollout_once(disease, seed_base + k)
        logps_per_rollout.append(lp)
        rewards_per_rollout.append(r)
    
    r_t = torch.tensor(rewards_per_rollout, device=model.device)
    advantage = (r_t - r_t.mean()) / (r_t.std() + 1e-8)
    
    loss = torch.tensor(0.0, device=model.device, requires_grad=True)
    for i, lp_list in enumerate(logps_per_rollout):
        if not lp_list:
            continue
        loss = loss + (- advantage[i] * torch.stack(lp_list).sum())
    loss = loss / max(1, len(logps_per_rollout))
    
    optim.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
    optim.step()
    
    mean_r = statistics.mean(rewards_per_rollout)
    std_r  = statistics.stdev(rewards_per_rollout) if len(rewards_per_rollout) > 1 else 0.0
    max_r  = max(rewards_per_rollout)
    history.append({"step": step, "disease": disease, "mean_r": mean_r, "max_r": max_r, "loss": float(loss)})
    with log_path.open("a", newline="") as fh:
        csv.writer(fh).writerow([step, disease, f"{mean_r:.4f}", f"{std_r:.4f}", f"{max_r:.4f}", f"{float(loss):.4f}"])
    if WANDB_API_KEY:
        wandb.log({"grpo/mean_reward": mean_r, "grpo/max_reward": max_r, "grpo/loss": float(loss), "step": step})
    print(f"step {step:>3}/{TOTAL_STEPS}  disease={disease:<22}  mean_r={mean_r:+.3f}  max_r={max_r:+.3f}  loss={float(loss):+.4f}")

## 6. Plot the reward curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(history)
df["mean_r_smooth"] = df["mean_r"].rolling(5, min_periods=1).mean()

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(df["step"], df["mean_r"], alpha=0.4, label="per-step mean")
ax.plot(df["step"], df["mean_r_smooth"], linewidth=2, label="5-step rolling mean")
ax.axhline(y=2.5, linestyle="--", color="green", label="oracle ceiling ≈ 2.5")
ax.set_xlabel("GRPO step")
ax.set_ylabel("group mean reward")
ax.set_title("GRPO reward curve (Colab mini)")
ax.legend()
plt.tight_layout()
plt.savefig("grpo_reward_curve.png", dpi=120)
plt.show()

## 7. Push the adapter

In [ ]:
if HF_TOKEN:
    model.push_to_hub(OUTPUT_HUB_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(OUTPUT_HUB_ID, token=HF_TOKEN)
    print(f"✓ pushed to https://huggingface.co/{OUTPUT_HUB_ID}")
else:
    model.save_pretrained("outputs/grpo-mini/final")
    tokenizer.save_pretrained("outputs/grpo-mini/final")
    print("✓ saved locally to outputs/grpo-mini/final")

if WANDB_API_KEY:
    wandb.finish()

## 8. Production checkpoint

The full GRPO run logged to W&B used 100 steps × group_size 8 × 3-level curriculum on an A100. Final adapter:

* **`SnehShah/house-md-grpo-optimized-gemma3-4b-v3`** — final
* **`SnehShah/house-md-grpo-optimized-gemma3-4b-step10`** — mid-training checkpoint

W&B dashboard: <https://wandb.ai/sneh2909-christ-university/house-md?nw=nwusersneh2909>

Notebook **`04_eval_compare.ipynb`** scores all three (base / SFT / GRPO) on the held-out 45-patient eval set.